# 예제 franka_ex06: FR3 그리퍼 제어 — Franka Hand

Franka FR3 의 손(`franka_hand`) 그리퍼를 직접 열고 닫는 예제.
6-DOF 용 `ex06_gripper_control.py` 를 FR3 환경에 맞춰 옮겨온 self-contained 노트북이다.

**6-DOF 예제와 다른 점**
- 그리퍼 컨트롤러: `fr3_gripper_controller` — `JointTrajectoryController` 타입 (6-DOF 예제는 `GripperActionController` 였음)
- 그래서 인터페이스도 `GripperCommand` 액션이 아니라 `FollowJointTrajectory` 액션 (`/fr3_gripper_controller/follow_joint_trajectory`) 으로 바뀐다
- 액티브 조인트는 `fr3_finger_joint1` (prismatic, 0.0=닫힘 ~ 0.04=한쪽 손가락 최대 열림)
- `fr3_finger_joint2` 는 URDF `<mimic>` 으로 따라옴 — 컨트롤러 명령에는 안 들어감
- Gazebo Sim 환경이므로 `use_sim_time=True`

**학습 내용**
- `FollowJointTrajectory` 액션의 기본 사용법 (`JointTrajectory` + `JointTrajectoryPoint`)
- prismatic 조인트와 mimic 조인트
- 손가락 위치(m) 단위와 실제 손가락 사이 갭 (한쪽 0.04m → 양쪽 합계 ~8cm 갭)
- `time_from_start` 으로 동작 속도 조절

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `(이 노트북은 마커 발행 없음 — RViz는 모델 시각화용으로만 사용)` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 상수 정의

In [1]:
# FR3 gripper: fr3_gripper_controller (JointTrajectoryController)
# 액티브 조인트는 fr3_finger_joint1 (prismatic, 0.0=닫힘, 0.04=한쪽 손가락 최대 열림).
# fr3_finger_joint2 는 mimic 이라 컨트롤러에는 안 들어감.
GRIPPER_JOINT     = 'fr3_finger_joint1'
GRIPPER_ACTION    = '/fr3_gripper_controller/follow_joint_trajectory'
GRIPPER_OPEN      = 0.04   # 최대 열림 (m)
GRIPPER_CLOSED    = 0.0    # 닫힘 (m)

## 2. ROS 2 초기화

`use_sim_time=True` 로 Gazebo Sim 시계와 맞춘다.

In [2]:
import rclpy
from rclpy.node import Node
from rclpy.parameter import Parameter
from rclpy.action import ActionClient

from control_msgs.action import FollowJointTrajectory
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from builtin_interfaces.msg import Duration

In [3]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

### 2-1. 노드 + `FollowJointTrajectory` 액션 클라이언트

In [4]:
node = Node(
    'franka_ex06_gripper_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
gripper_client = ActionClient(node, FollowJointTrajectory, GRIPPER_ACTION)
node.get_logger().info('=== franka_ex06 노트북 노드 생성 완료 ===')

[INFO] [1783238752.477282247] [franka_ex06_gripper_demo]: === franka_ex06 노트북 노드 생성 완료 ===


True

## 3. 그리퍼 액션 서버 대기

Gazebo + ros2_control 의 `fr3_gripper_controller` 가 spawn 되어 액션 서버를 띄울 때까지 기다린다.

In [5]:
if not gripper_client.wait_for_server(timeout_sec=30.0):
    raise RuntimeError(f'gripper 액션 서버({GRIPPER_ACTION}) 연결 실패')
node.get_logger().info('gripper 서버 준비됨')

[INFO] [1783238754.919634219] [franka_ex06_gripper_demo]: gripper 서버 준비됨


True

## 4. 그리퍼 명령 헬퍼

`FollowJointTrajectory` 는 그리퍼처럼 단일 점 이동에도 쓸 수 있다.
`JointTrajectoryPoint` 한 개에 목표 위치(`positions=[pos]`) 와 도달 시간(`time_from_start`) 만 채운다.

`fr3_finger_joint1` 1개만 보낸다 (`fr3_finger_joint2` 는 URDF `<mimic>` 으로 따라옴).

In [6]:
def move_gripper(position: float, duration_sec: float = 1.0) -> bool:
    '''fr3_finger_joint1 을 `position` (m) 으로 이동.
    position 은 [0.0, 0.04] 로 클램프 한다.'''
    pos = float(max(0.0, min(GRIPPER_OPEN, position)))

    goal = FollowJointTrajectory.Goal()
    goal.trajectory = JointTrajectory()
    goal.trajectory.joint_names = [GRIPPER_JOINT]
    pt = JointTrajectoryPoint()
    pt.positions = [pos]
    pt.time_from_start = Duration(
        sec=int(duration_sec),
        nanosec=int((duration_sec - int(duration_sec)) * 1e9),
    )
    goal.trajectory.points.append(pt)

    send_future = gripper_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, send_future)
    handle = send_future.result()
    if handle is None or not handle.accepted:
        node.get_logger().error('gripper goal 거부')
        return False

    result_future = handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)
    code_val = result_future.result().result.error_code
    ok = (code_val == 0)  # FollowJointTrajectory.Result.SUCCESSFUL = 0
    node.get_logger().info(
        f'  → {pos*1000:.1f}mm 이동 완료 (error_code={code_val}, ok={ok})'
    )
    return ok

def open_gripper():  return move_gripper(GRIPPER_OPEN)
def close_gripper(): return move_gripper(GRIPPER_CLOSED)

## 5. 시연 — 열기 / 닫기 / 절반 / 다시 열기

각 단계 사이에 1초 정도 쉬어 RViz / Gazebo 에서 손가락이 움직이는 걸 눈으로 확인한다.

### 5-1. 열기 (0.04m, 한쪽 4cm 까지 벌림)

In [13]:
import time
node.get_logger().info('--- 1단계: 그리퍼 열기 ---')
open_gripper()
time.sleep(1.5)

[INFO] [1783238787.601510835] [franka_ex06_gripper_demo]: --- 1단계: 그리퍼 열기 ---
[INFO] [1783238788.604617597] [franka_ex06_gripper_demo]:   → 40.0mm 이동 완료 (error_code=0, ok=True)


### 5-2. 닫기 (0.0m)

In [11]:
node.get_logger().info('--- 2단계: 그리퍼 닫기 ---')
close_gripper()
time.sleep(1.5)

[INFO] [1783238781.985676519] [franka_ex06_gripper_demo]: --- 2단계: 그리퍼 닫기 ---
[INFO] [1783238782.988252483] [franka_ex06_gripper_demo]:   → 0.0mm 이동 완료 (error_code=0, ok=True)


### 5-3. 절반 열기 (0.02m)

In [12]:
node.get_logger().info('--- 3단계: 그리퍼 절반 열기 ---')
move_gripper(0.02)
time.sleep(1.5)

[INFO] [1783238784.740086221] [franka_ex06_gripper_demo]: --- 3단계: 그리퍼 절반 열기 ---
[INFO] [1783238785.743053002] [franka_ex06_gripper_demo]:   → 20.0mm 이동 완료 (error_code=0, ok=True)


### 5-4. 다시 완전 열기

In [14]:
node.get_logger().info('--- 4단계: 다시 그리퍼 열기 ---')
open_gripper()
time.sleep(1.0)
node.get_logger().info('=== franka_ex06 완료! ===')

[INFO] [1783238790.832665994] [franka_ex06_gripper_demo]: --- 4단계: 다시 그리퍼 열기 ---
[INFO] [1783238791.835902067] [franka_ex06_gripper_demo]:   → 40.0mm 이동 완료 (error_code=0, ok=True)
[INFO] [1783238792.838561785] [franka_ex06_gripper_demo]: === franka_ex06 완료! ===


True

## 6. 보너스 — 빠르게/천천히 동작 비교

`duration_sec` 을 바꾸면 도달 시간이 달라진다.
컨트롤러는 시작점→끝점 사이를 보간하므로 `duration_sec` 이 작을수록 빠르게 움직인다.

In [15]:
node.get_logger().info('빠르게 닫기 (0.3초)')
move_gripper(GRIPPER_CLOSED, duration_sec=0.3)
time.sleep(1.0)

node.get_logger().info('천천히 열기 (3초)')
move_gripper(GRIPPER_OPEN, duration_sec=3.0)
time.sleep(3.5)

[INFO] [1783238793.872663807] [franka_ex06_gripper_demo]: 빠르게 닫기 (0.3초)
[INFO] [1783238794.175520274] [franka_ex06_gripper_demo]:   → 0.0mm 이동 완료 (error_code=0, ok=True)
[INFO] [1783238795.178024431] [franka_ex06_gripper_demo]: 천천히 열기 (3초)
[INFO] [1783238798.181072397] [franka_ex06_gripper_demo]:   → 40.0mm 이동 완료 (error_code=0, ok=True)


## 7. 정리

In [16]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass